# 01 HumAID: load, inspect, baseline

Member 5. Loads HumAID (`QCRI/HumAID-all` on Hugging Face), checks its structure, saves a copy to Drive, and trains a TF-IDF + logistic regression baseline to compare the transformer classifier against.

Runs on CPU. Needs internet access for the first download only. Clear outputs before committing. Keep the data on Drive (it is never committed).
Licence: CrisisNLP terms of use, https://crisisnlp.qcri.org/terms-of-use.html. Add a row for it to `data/README.md`.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets", "scikit-learn", "pyarrow"], check=True)

import os
from google.colab import drive
drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
OUT = f"{ROOT}/humaid"
os.makedirs(OUT, exist_ok=True)

HF_NAME = "QCRI/HumAID-all"
SEED = 42

## Load and inspect

In [ ]:
from datasets import load_dataset

ds = load_dataset(HF_NAME, verification_mode="no_checks")
print(ds)
for name, split in ds.items():
    print(name, split.num_rows, split.column_names)
first = list(ds.keys())[0]
print(ds[first][0])

The next cell finds the text and label columns by name. If it stops with an error, read the column list printed above and set `TEXT_COL` and `LABEL_COL` by hand.

In [ ]:
import pandas as pd
import numpy as np

TEXT_COL, LABEL_COL = None, None     # leave as None to detect


def find_columns(split, text_col=None, label_col=None):
    """Text and label column names, from the candidates below unless given."""
    cols = split.column_names
    text = text_col or next((c for c in ("tweet_text", "text", "tweet") if c in cols), None)
    label = label_col or next((c for c in ("class_label", "label", "labels") if c in cols), None)
    if text is None or label is None:
        raise ValueError(f"Could not find the columns in {cols}. Set TEXT_COL and LABEL_COL by hand.")
    return text, label


def to_frame(split, text_col, label_col):
    """DataFrame with columns text and label. Class ids become class names when the split has them."""
    df = split.to_pandas()[[text_col, label_col]].rename(columns={text_col: "text", label_col: "label"})
    feat = split.features[label_col]
    if hasattr(feat, "names"):
        df["label"] = df["label"].map(dict(enumerate(feat.names)))
    df["label"] = df["label"].astype(str)
    return df


TEXT_COL, LABEL_COL = find_columns(ds[first], TEXT_COL, LABEL_COL)
frames = {name: to_frame(split, TEXT_COL, LABEL_COL) for name, split in ds.items()}
print("text:", TEXT_COL, "| label:", LABEL_COL)
print({k: len(v) for k, v in frames.items()})

### Labels

In [ ]:
import matplotlib.pyplot as plt


def label_counts(frames):
    """Tweets per class and split, as a table."""
    return pd.DataFrame({name: df["label"].value_counts() for name, df in frames.items()}).fillna(0).astype(int)


def plot_label_counts(counts, title):
    """Horizontal bars of the total number of tweets per class."""
    fig, ax = plt.subplots(figsize=(7, 4.5))
    counts.sum(axis=1).sort_values().plot.barh(ax=ax)
    ax.set_title(title)
    ax.set_xlabel("tweets")
    fig.tight_layout()
    return fig


counts = label_counts(frames)
print(counts)
plot_label_counts(counts, "HumAID tweets per class");

### Text length and leakage checks

In [ ]:
def length_stats(df):
    """Words per tweet: mean, median, 95th percentile and maximum."""
    n = df["text"].str.split().str.len()
    return {"mean": round(float(n.mean()), 1), "median": float(n.median()),
            "p95": float(n.quantile(0.95)), "max": int(n.max())}


def plot_lengths(df, title):
    """Histogram of words per tweet."""
    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.hist(df["text"].str.split().str.len(), bins=40)
    ax.set_title(title)
    ax.set_xlabel("words")
    fig.tight_layout()
    return fig


def text_overlap(a, b):
    """Number of tweets in b whose exact text also appears in a (leakage between splits)."""
    return int(b["text"].isin(set(a["text"])).sum())


train_name = "train" if "train" in frames else first
print(train_name, length_stats(frames[train_name]))
for name in frames:
    if name != train_name:
        print(f"{name}: {text_overlap(frames[train_name], frames[name])} of {len(frames[name])} tweets also appear in {train_name}")
plot_lengths(frames[train_name], "Words per tweet (train)");

## Save a copy to Drive

In [ ]:
def save_frames(frames, out_dir):
    """One parquet per split with columns text and label, so later notebooks do not download again."""
    for name, df in frames.items():
        df.to_parquet(f"{out_dir}/{name}.parquet", index=False)
        print("saved", name, len(df))


save_frames(frames, OUT)

## Baseline: TF-IDF + logistic regression

A simple model to beat. It also shows how hard the task is before any fine-tuning. Scores are macro-F1 (every class counts the same), plus accuracy.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline


def pick_split(frames, *names):
    """First split name that exists, e.g. validation or dev."""
    return next((n for n in names if n in frames), None)


def train_baseline(train_df, seed=SEED):
    """TF-IDF (word 1-2 grams) with a class-balanced logistic regression."""
    model = make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=True),
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=seed))
    return model.fit(train_df["text"], train_df["label"])


def evaluate(model, df):
    """Accuracy, macro-F1 and the predictions for one split."""
    pred = model.predict(df["text"])
    return {"accuracy": round(accuracy_score(df["label"], pred), 3),
            "macro_f1": round(f1_score(df["label"], pred, average="macro"), 3)}, pred


baseline = train_baseline(frames[train_name])
val_name = pick_split(frames, "validation", "dev", "val")
test_name = pick_split(frames, "test")
scores = {}
for name in (val_name, test_name):
    if name:
        scores[name], _ = evaluate(baseline, frames[name])
pd.DataFrame(scores).T

In [ ]:
def per_class_report(df, pred):
    """Precision, recall and F1 per class as a table."""
    rep = classification_report(df["label"], pred, output_dict=True, zero_division=0)
    return pd.DataFrame(rep).T.round(3)


def plot_confusion(df, pred, title):
    """Confusion matrix with row-normalised colours and counts in the cells."""
    labels = sorted(df["label"].unique())
    cm = confusion_matrix(df["label"], pred, labels=labels)
    norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels)), labels, rotation=60, ha="right")
    ax.set_yticks(range(len(labels)), labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(title)
    fig.tight_layout()
    return fig


eval_name = test_name or val_name
_, pred = evaluate(baseline, frames[eval_name])
print(per_class_report(frames[eval_name], pred))
plot_confusion(frames[eval_name], pred, f"TF-IDF baseline on {eval_name}");

## Notes for the report
- Report the baseline's macro-F1 next to the fine-tuned transformer's, on the same test split.
- HumAID has English tweets only. The Arabic advisory is generated text, not a classifier trained on Arabic, so say that in the limits.
- Decide with the team what to do with the "not humanitarian", "other relevant information" and "dont know cant judge" classes before the transformer run, and use the same choice for the baseline.
- If `text_overlap` is above zero for the test split, remove the duplicates from the test split before reporting scores.